## Problem: Save and Load Your PyTorch Model

### Problem Statement
You are tasked with saving a trained PyTorch model to a file and later loading it for inference or further training. This process allows you to persist the trained model and use it in different environments without retraining.

### Requirements
1. **Save the Model**:
   - Save the model’s **state dictionary** (weights) to a file named `model.pth` using `torch.save`.

2. **Load the Model for Inference**:
   - Create a fresh `SimpleModel`, load the saved state dictionary with `torch.load(..., map_location="cpu")` and `load_state_dict`, and switch it to evaluation mode.
   - The loaded model must produce exactly the same predictions as the original.

3. **Checkpoint to Resume Training**:
   - Save a dictionary holding the epoch, the model's state dict **and the optimizer's state dict** to `checkpoint.pth`.
   - Restore both into a fresh model and optimizer, so training could continue where it stopped.

### Notes
- Save `state_dict()` rather than `torch.save(model)`: pickling the whole module ties the file to your class's code and import path, while a state dict is just a mapping from parameter names to tensors.
- `map_location` lets a checkpoint written on a GPU be loaded on a CPU-only machine (or onto a different device).
- Since PyTorch 2.6, `torch.load` defaults to `weights_only=True`, which refuses to unpickle arbitrary Python objects; state dicts and plain dicts of tensors and numbers still load.
- `model.eval()` switches layers such as dropout and batch norm to inference behaviour. This model has neither, but forgetting it is a classic bug.
- Resuming training needs the optimizer state too — here SGD's momentum buffers (for Adam, the first and second moment estimates). Weights alone are not enough.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        return self.fc(x)

# Create and train the model
torch.manual_seed(42)
model = SimpleModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Training loop
X = torch.rand(100, 1)
y = 3 * X + 2 + torch.randn(100, 1) * 0.1
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, y)
    loss.backward()
    optimizer.step()

In [ ]:
# Save the model's state_dict to a file named "model.pth"
torch.save(model.state_dict(), "model.pth")

# Load it into a fresh SimpleModel and switch to eval mode
loaded_model = SimpleModel()
loaded_model.load_state_dict(torch.load("model.pth", map_location="cpu", weights_only=True))
loaded_model.eval()

In [ ]:
# Save a training checkpoint: everything needed to continue training later
torch.save({
    "epoch": epochs,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "checkpoint.pth")

# Restore the checkpoint into a fresh model and optimizer
resumed_model = SimpleModel()
resumed_optimizer = optim.SGD(resumed_model.parameters(), lr=0.01, momentum=0.9)
checkpoint = torch.load("checkpoint.pth", map_location="cpu", weights_only=True)
resumed_model.load_state_dict(checkpoint["model_state_dict"])
resumed_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]
resumed_model.train()  # training would continue from start_epoch

In [ ]:
# Verify the model works after loading
X_test = torch.tensor([[0.5], [1.0], [1.5]])
with torch.no_grad():
    predictions = loaded_model(X_test)
    print(f"Predictions after loading: {predictions}")
    assert torch.equal(predictions, model(X_test)), "the loaded model disagrees with the original"
    assert torch.equal(resumed_model(X_test), model(X_test)), "the resumed model disagrees with the original"

# The optimizer state came back too: SGD's momentum buffers match the original's
for p_orig, p_new in zip(model.parameters(), resumed_model.parameters()):
    assert torch.equal(optimizer.state[p_orig]["momentum_buffer"],
                       resumed_optimizer.state[p_new]["momentum_buffer"])
assert start_epoch == epochs
print("Weights and optimizer state restored exactly.")